# Task 3：Synthetic Phishing Data Generation

NTHU 114 學年第 2 學期 — LLM Security System — Week 6

> 本作品為與 AI 共同協作的成果。

使用 LLM 生成合成釣魚信件資料，沿用 Task 1 設計的 schema，擴充資安訓練資料集。

## Cell 0：環境設定

In [1]:
# Cell 0：環境設定
import json
import csv
import re
from pathlib import Path

import litellm
import pandas as pd

# ===== LLM 設定（同 main.ipynb）=====
USE_LOCAL_LLM  = True

if USE_LOCAL_LLM:
    LLM_MODEL    = "ollama/gpt-oss:20b"
    LLM_API_BASE = "http://localhost:11434"
else:
    LLM_MODEL    = "gemini/gemini-2.5-flash"
    LLM_API_BASE = None

LLM_TEMPERATURE = 0.8   # 稍高溫度以增加多樣性
LLM_MAX_TOKENS  = 8192  # 15 筆 JSON 需要較大的輸出空間

OUTPUT_DIR = Path("data")

print(f"LLM 模式：{'本地 Ollama' if USE_LOCAL_LLM else '雲端 API'}（{LLM_MODEL}）")
print(f"Temperature: {LLM_TEMPERATURE}")
print(f"Max tokens: {LLM_MAX_TOKENS}")

LLM 模式：本地 Ollama（ollama/gpt-oss:20b）
Temperature: 0.8
Max tokens: 8192


## Cell 1：定義生成 Prompt

使用 Task 3 設計的 prompt，指示 LLM 生成 15 筆符合 schema 的合成釣魚信件。

In [2]:
# Cell 1：定義生成 Prompt
GENERATION_PROMPT = """你是一位資安研究員，正在建立釣魚信件偵測資料集。請根據以下 schema 生成 15 筆合成釣魚信件資料（JSON 格式）。

Schema 欄位：
- email_id: 字串，格式為 "SYNTH-001" 遞增
- sender: 偽造的寄件者地址（要模擬真實釣魚手法，如拼寫相似的假網域）
- subject: 信件主旨
- body: 信件內文（50-150 字）
- urls: 信件中的惡意連結陣列（可為空）
- has_attachment: 布林值
- urgency_level: "high" / "medium" / "low"
- label: 固定為 "phishing"
- attack_type: 從以下選擇：credential_harvesting, BEC, malware_delivery, invoice_fraud
- indicators: 描述為什麼這封信是釣魚信的關鍵指標

要求：
1. 攻擊類型要多樣化，至少涵蓋 3 種 attack_type
2. 包含不同語言的信件（英文、中文、日文各至少 1 封）
3. urgency_level 要有高中低的分佈
4. 偽造的網域要有創意（不要只用數字替換字母）
5. 每封信的社交工程手法要不同

請只回傳 JSON 陣列，不要包含任何其他文字或 markdown 標記。"""

print("Prompt 已定義")
print(f"Prompt 長度：{len(GENERATION_PROMPT)} 字元")

Prompt 已定義
Prompt 長度：588 字元


## Cell 2：呼叫 LLM 生成資料

In [3]:
# Cell 2：呼叫 LLM 生成合成資料
print("正在呼叫 LLM 生成 15 筆合成釣魚信件...")

response = litellm.completion(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": GENERATION_PROMPT}],
    temperature=LLM_TEMPERATURE,
    max_tokens=LLM_MAX_TOKENS,
    **({"api_base": LLM_API_BASE} if LLM_API_BASE else {}),
)

raw_output = response.choices[0].message.content
print(f"LLM 回應長度：{len(raw_output)} 字元")
print("---")
print(raw_output[:500] + "..." if len(raw_output) > 500 else raw_output)

正在呼叫 LLM 生成 15 筆合成釣魚信件...
LLM 回應長度：7126 字元
---
[
  {
    "email_id": "SYNTH-001",
    "sender": "support@gmaill.com",
    "subject": "Urgent: Password Reset Required",
    "body": "Dear user, our system detected unusual activity on your account. Please click the link below to reset your password immediately to avoid suspension.",
    "urls": ["https://secure.gmaill.com/reset?uid=12345"],
    "has_attachment": false,
    "urgency_level": "high",
    "label": "phishing",
    "attack_type": "credential_harvesting",
    "indicators": "The email ...


## Cell 3：解析 JSON 並驗證 Schema

In [4]:
# Cell 3：解析 JSON 並驗證 Schema

# --- 3a. 解析 JSON ---
# LLM 可能在 JSON 前後加上 markdown 標記，需要清理
cleaned = raw_output.strip()
if cleaned.startswith("```"):
    # 移除 ```json ... ``` 包裹
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

synthetic_data = json.loads(cleaned)
print(f"成功解析 {len(synthetic_data)} 筆資料\n")

# --- 3b. Schema 驗證 ---
REQUIRED_FIELDS = {
    "email_id": str,
    "sender": str,
    "subject": str,
    "body": str,
    "urls": list,
    "has_attachment": bool,
    "urgency_level": str,
    "label": str,
    "attack_type": str,
    "indicators": str,
}
VALID_URGENCY = {"high", "medium", "low"}
VALID_ATTACK = {"credential_harvesting", "BEC", "malware_delivery", "invoice_fraud"}
EMAIL_PATTERN = re.compile(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")

issues = []
for entry in synthetic_data:
    eid = entry.get("email_id", "UNKNOWN")

    # 檢查必要欄位存在與型別
    for field, expected_type in REQUIRED_FIELDS.items():
        if field not in entry:
            issues.append(f"{eid}: 缺少欄位 '{field}'")
        elif not isinstance(entry[field], expected_type):
            issues.append(f"{eid}: '{field}' 型別錯誤（期望 {expected_type.__name__}，實際 {type(entry[field]).__name__}）")

    # 檢查 urgency_level
    if entry.get("urgency_level") not in VALID_URGENCY:
        issues.append(f"{eid}: urgency_level '{entry.get('urgency_level')}' 不在有效範圍")

    # 檢查 attack_type
    if entry.get("attack_type") not in VALID_ATTACK:
        issues.append(f"{eid}: attack_type '{entry.get('attack_type')}' 不在有效範圍")

    # 檢查 email 格式（RFC 5321）
    sender = entry.get("sender", "")
    if not EMAIL_PATTERN.match(sender):
        issues.append(f"{eid}: sender '{sender}' 不符合標準 email 格式（可能包含非 ASCII 字元）")

    # 檢查 label
    if entry.get("label") != "phishing":
        issues.append(f"{eid}: label 應為 'phishing'，實際為 '{entry.get('label')}'")

if issues:
    print(f"⚠️ 發現 {len(issues)} 個問題：")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("✅ 所有 15 筆資料通過 schema 驗證！")

成功解析 15 筆資料

✅ 所有 15 筆資料通過 schema 驗證！


## Cell 4：儲存為 JSON + CSV

In [5]:
# Cell 4：儲存為 JSON + CSV

json_path = OUTPUT_DIR / "synthetic_phishing_dataset.json"
csv_path  = OUTPUT_DIR / "synthetic_phishing_dataset.csv"

# --- 4a. 儲存 JSON ---
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(synthetic_data, f, ensure_ascii=False, indent=2)
print(f"✅ JSON 已儲存：{json_path}")

# --- 4b. 儲存 CSV ---
df = pd.DataFrame(synthetic_data)
# urls 欄位從 list 轉為分號分隔字串（與現有 CSV 格式一致）
df["urls"] = df["urls"].apply(lambda x: ";".join(x) if isinstance(x, list) else x)
df.to_csv(csv_path, index=False, encoding="utf-8")
print(f"✅ CSV 已儲存：{csv_path}")

# --- 4c. 統計摘要 ---
print(f"\n===== 統計摘要 =====")
print(f"總筆數：{len(synthetic_data)}")
print(f"\n攻擊類型分佈：")
print(df["attack_type"].value_counts().to_string())
print(f"\n急迫程度分佈：")
print(df["urgency_level"].value_counts().to_string())
print(f"\n有附件：{df['has_attachment'].sum()} 筆")
print(f"含惡意連結：{(df['urls'] != '').sum()} 筆")

✅ JSON 已儲存：data/synthetic_phishing_dataset.json
✅ CSV 已儲存：data/synthetic_phishing_dataset.csv

===== 統計摘要 =====
總筆數：15

攻擊類型分佈：
attack_type
credential_harvesting    4
BEC                      4
malware_delivery         4
invoice_fraud            3

急迫程度分佈：
urgency_level
high      5
medium    5
low       5

有附件：7 筆
含惡意連結：11 筆


## Cell 5：品質分析

In [6]:
# Cell 5：品質分析

print("===== 品質分析 =====")
print()

# 多樣性檢查
attack_types = set(df["attack_type"])
urgency_levels = set(df["urgency_level"])
print(f"攻擊類型涵蓋：{len(attack_types)}/4 種 — {', '.join(sorted(attack_types))}")
print(f"急迫程度涵蓋：{len(urgency_levels)}/3 種 — {', '.join(sorted(urgency_levels))}")

# 語言多樣性（簡單偵測）
def detect_lang(text):
    if re.search(r'[\u4e00-\u9fff]', text):
        return '中文'
    if re.search(r'[\u3040-\u309f\u30a0-\u30ff]', text):
        return '日文'
    return '英文'

langs = [detect_lang(entry["body"]) for entry in synthetic_data]
lang_counts = pd.Series(langs).value_counts()
print(f"語言分佈：")
for lang, count in lang_counts.items():
    print(f"  {lang}: {count} 筆")

# Email 格式問題統計
bad_emails = [e["email_id"] for e in synthetic_data if not EMAIL_PATTERN.match(e.get("sender", ""))]
if bad_emails:
    print(f"\n⚠️ Email 格式不合規的筆數：{len(bad_emails)} — {', '.join(bad_emails)}")
    print("（LLM 常見錯誤：將顯示名稱混入 email 地址，或使用非 ASCII 字元）")
else:
    print(f"\n✅ 所有 sender 欄位皆符合 RFC 5321 email 格式")

# 逐筆預覽
print(f"\n===== 逐筆預覽 =====")
for entry in synthetic_data:
    print(f"\n[{entry['email_id']}] {entry['attack_type']} | {entry['urgency_level']}")
    print(f"  From: {entry['sender']}")
    print(f"  Subject: {entry['subject']}")
    print(f"  Body: {entry['body'][:80]}..." if len(entry['body']) > 80 else f"  Body: {entry['body']}")

===== 品質分析 =====

攻擊類型涵蓋：4/4 種 — BEC, credential_harvesting, invoice_fraud, malware_delivery
急迫程度涵蓋：3/3 種 — high, low, medium
語言分佈：
  英文: 8 筆
  中文: 7 筆

✅ 所有 sender 欄位皆符合 RFC 5321 email 格式

===== 逐筆預覽 =====

[SYNTH-001] credential_harvesting | high
  From: support@gmaill.com
  Subject: Urgent: Password Reset Required
  Body: Dear user, our system detected unusual activity on your account. Please click th...

[SYNTH-002] BEC | medium
  From: ceo@twiter.com
  Subject: Immediate Approval Needed for Invoice #9876
  Body: Hi team, I need the attached invoice approved by EOD. Please review and sign off...

[SYNTH-003] malware_delivery | low
  From: updates@amaz0n.com
  Subject: Important System Update Available
  Body: Please download the latest driver from the link below to keep your device secure...

[SYNTH-004] invoice_fraud | high
  From: billing@paypa1.com
  Subject: 您的發票已準備好 - 付款請盡快
  Body: 尊敬的客戶，您的發票已生成，請點擊下方連結完成付款，以免延遲。感謝您的合作！

[SYNTH-005] credential_harvesting | medium
  From: login